# Few-shot 영화 추천 챗봇 실습

목표: `ratings_train.txt` 영화 리뷰 데이터를 이용해 사용자 취향과 비슷한 리뷰를 찾고, few-shot 예제와 함께 추천 프롬프트를 구성한다.

이번 노트북은 수업 문서 `few_shot_추천챗봇.docx`의 흐름을 코드로 재구성한 것이다.

## STEP 1. 데이터 불러오기

In [ ]:
from pathlib import Path
import csv

DATA_PATH = Path('../data/ratings_train.txt')
rows = []

with DATA_PATH.open('r', encoding='utf-8') as file:
    reader = csv.DictReader(file, delimiter='\t')
    for row in reader:
        document = (row.get('document') or '').strip()
        if len(document) < 5:
            continue
        rows.append({
            'id': row.get('id'),
            'document': document,
            'label': int(row.get('label') or 0),
        })

len(rows), rows[:3]

## STEP 2. 간단한 한국어 토큰화

수업 목적상 형태소 분석기 없이 정규식과 간단한 조사 제거만 사용한다.

In [ ]:
import re

STOPWORDS = {'영화', '진짜', '너무', '정말', '보고', '있는', '없는', '그냥'}

def tokenize(text):
    text = re.sub(r'[^0-9A-Za-z가-힣\s]', ' ', text.lower())
    raw_tokens = re.findall(r'[0-9A-Za-z가-힣]{2,}', text)
    tokens = []
    for token in raw_tokens:
        normalized = re.sub(r'(은|는|이|가|을|를|에|의|도|만|로|으로)$', '', token)
        if len(normalized) >= 2 and normalized not in STOPWORDS:
            tokens.append(normalized)
    return tokens

tokenize('로맨틱한 영화가 좋아요. 감정적으로 감동하는 느낌을 원해요')

## STEP 3. TF-IDF 인덱스 만들기

전체 15만 건을 모두 써도 되지만, 실습 속도를 위해 처음에는 3만 건만 사용한다.

In [ ]:
from collections import Counter, defaultdict
import math

sample_rows = rows[:30000]
tokenized = []
doc_freq = defaultdict(int)

for row in sample_rows:
    tokens = tokenize(row['document'])
    if not tokens:
        continue
    tokenized.append({**row, 'tokens': tokens})
    for token in set(tokens):
        doc_freq[token] += 1

total_docs = len(tokenized)
idf = {token: math.log((1 + total_docs) / (1 + count)) + 1 for token, count in doc_freq.items()}

def tfidf(tokens):
    counts = Counter(tokens)
    vector = {token: (count / len(tokens)) * idf.get(token, 0.0) for token, count in counts.items()}
    norm = math.sqrt(sum(weight * weight for weight in vector.values())) or 1.0
    return {token: weight / norm for token, weight in vector.items()}

for row in tokenized:
    row['vector'] = tfidf(row['tokens'])

len(tokenized), list(idf.items())[:5]

## STEP 4. 유사 리뷰 검색

In [ ]:
def cosine(query_vector, doc_vector):
    return sum(weight * doc_vector.get(token, 0.0) for token, weight in query_vector.items())

def search_reviews(query, top_k=5):
    query_tokens = tokenize(query)
    query_vector = tfidf(query_tokens)
    scored = []
    for row in tokenized:
        score = cosine(query_vector, row['vector'])
        if score > 0:
            scored.append((score, row))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [
        {
            'score': round(score, 4),
            'label': '긍정' if row['label'] == 1 else '부정',
            'document': row['document'],
            'matched_tokens': sorted(set(query_tokens) & set(row['tokens'])),
        }
        for score, row in scored[:top_k]
    ]

query = '로맨틱한 영화가 좋아요. 감정적으로 감동하는 느낌을 원해요'
search_reviews(query)

## STEP 5. Few-shot 프롬프트 구성

In [ ]:
few_shot_examples = [
    {'label': '긍정', 'review': '액션이 없는데도 재미 있는 몇 안되는 영화', 'analysis': '느린 전개라도 몰입감과 감정선이 좋으면 긍정적으로 평가한다.'},
    {'label': '부정', 'review': '아 더빙.. 진짜 짜증나네요 목소리', 'analysis': '연기, 더빙, 몰입 방해 요소가 있으면 부정적으로 평가한다.'},
    {'label': '중립', 'review': '담백하고 깔끔해서 좋다. 신문기사로만 보다 직접 보니 느낌이 다르다.', 'analysis': '강한 재미보다 담백한 분위기와 현실감 있는 구성을 선호한다.'},
]

def build_prompt(query, similar_reviews):
    examples = '\n'.join([f"- 예제({item['label']}): \"{item['review']}\" -> {item['analysis']}" for item in few_shot_examples])
    reviews = '\n'.join([f"{idx + 1}. ({item['label']}, score={item['score']}) {item['document']}" for idx, item in enumerate(similar_reviews)])
    return f'''당신은 영화 추천 전문가입니다.

【Few-shot 예제】
{examples}

【검색된 유사 리뷰】
{reviews}

【사용자 요청】
{query}

위의 예제와 참고 리뷰를 바탕으로 추천할 영화를 제시하고, 추천 이유를 상세히 설명해주세요.
'''

similar = search_reviews(query)
prompt = build_prompt(query, similar)
print(prompt)

## STEP 6. 다음 확장

- OpenAI API를 연결해 실제 추천 문장을 생성한다.
- 같은 로직을 FastAPI backend로 옮긴다.
- Next.js 화면에서 사용자 입력, 유사 리뷰, few-shot 예제, 최종 프롬프트를 한 번에 확인한다.